# Notebook 07: receita final na base de 339 features

**Unidade curricular:** Dados e Aprendizagem Automática (DAA), 2026/2027

Este notebook fecha a competição. Valida uma só vez, na base final do Notebook 02 com 339 features, a receita combinada: HGB, contagem de não-respostas, clip de outliers, stacking, escalas e correção pela idade mínima. Os Notebooks 04 e 05 ficam como evidência histórica da base antiga de 409 features.

Corre depois do Notebook 02 e demora cerca de 15 minutos.


## Carga e construção da base final

Carregamos a base nova do Notebook 02 e acrescentamos os dois contributos externos que sobreviveram à auditoria: a contagem de "Sem resposta" e o clip entre os percentis 0.5 e 99.5. Tudo o resto foi rejeitado com número, incluindo os blocos de features duplicados, as interações, os encodings alternativos e a afinação. `PID` é identificador e fica fora das features.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

plt.style.use('ggplot')
sns.set_theme(style='whitegrid')

DATA_DIR = '../../data' if os.path.isdir('../../data') else '../data'
SUBM_DIR = '../submissions'
TARGET = 'AGE_RANGE'
RANDOM_STATE = 42

train_nb02 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
test_nb02 = pd.read_csv(os.path.join(DATA_DIR, 'test_nb02.csv'))
train_nb01 = pd.read_csv(os.path.join(DATA_DIR, 'train_nb01.csv'), index_col='PID',
                           keep_default_na=False, na_values=[''])
test_nb01 = pd.read_csv(os.path.join(DATA_DIR, 'test_nb01.csv'), index_col='PID',
                          keep_default_na=False, na_values=[''])
assert len(train_nb02) == 10962 and len(test_nb02) == 3655
assert 'FAIXA_MINIMA' in train_nb02.columns, 'correr o NB02 novo primeiro'

y = train_nb02[TARGET]
X = train_nb02.drop(columns=['PID', TARGET])
Xte = test_nb02[X.columns]
fmin_tr = train_nb02['FAIXA_MINIMA'].values
fmin_te = test_nb02['FAIXA_MINIMA'].values
CLASSES = sorted(y.unique())
print('base:', X.shape)

FLOAT_CLIP = ['IN2', 'IN3', 'IN4', 'HHNBPERS_COD', 'HHNBPERS_0_13_COD', 'HH_ACT',
              'HH_INACT', 'HO12_COD', 'HO34_COD', 'AM3', 'AM5', 'IN24', 'IN25', 'IN26',
              'BM1', 'BM2', 'PE2', 'PE4', 'PE6', 'PE7', 'PE8', 'PE9', 'IN29', 'DH2',
              'DH4', 'SK2', 'IN45', 'IN47', 'IN55', 'IN56', 'IN57', 'IN59', 'PA1_MESES']
for c in [c for c in FLOAT_CLIP if c in X.columns]:
    for D in (X, Xte):
        vv = D[c]
        vv = vv[vv != -1]
        lo, hi = float(vv.quantile(0.005)), float(vv.quantile(0.995))
        m = D[c] != -1
        D.loc[m, c] = D.loc[m, c].clip(lo, hi)

for D, t in ((X, train_nb01), (Xte, test_nb01)):
    txt = t.select_dtypes(include='str')
    D['N_SEM_RESPOSTA'] = ((txt == 'Sem resposta') | (txt == 'NR')).sum(axis=1).values
print('features finais:', X.shape[1])

FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(X, y))
print('folds fixas: 5, seed 42 (as mesmas de NB04/05)')

## Verificação de sanidade e receita combinada

A verificação de sanidade treina o HGB com os valores por omissão e deve dar cerca de 0.467 na base nova. Se der outro valor, há um problema na construção dos dados.

A receita combinada tem cinco passos: treinar o classificador, treinar o regressor, escolher o limiar do stacking, ajustar as escalas por classe e aplicar a correção `max(previsão, FAIXA_MINIMA)`. Tudo é avaliado nas previsões out-of-fold, sem tocar no teste. O resultado esperado ronda 0.47. Abaixo de 0.475 não se submete e fica o melhor resultado confirmado.


In [ ]:
def oof_clf():
    oof = np.empty(len(y), dtype=int)
    proba = np.zeros((len(y), len(CLASSES)))
    for tr, va in FOLDS:
        m = HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_STATE)
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[va] = m.predict(X.iloc[va])
        proba[va] = m.predict_proba(X.iloc[va])
    return oof, proba


def oof_reg():
    o = np.empty(len(y))
    for tr, va in FOLDS:
        m = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
        m.fit(X.iloc[tr], y.iloc[tr])
        o[va] = m.predict(X.iloc[va])
    return np.clip(np.round(o), 0, 15).astype(int)


def tune_escalas(proba, yt, rondas=3):
    esc = np.ones(proba.shape[1])
    base = f1_score(yt, proba.argmax(axis=1), average='macro')
    for _ in range(rondas):
        for k in range(proba.shape[1]):
            for g in np.arange(0.5, 2.05, 0.25):
                t = esc.copy()
                t[k] = g
                aj = proba * t
                aj = aj / aj.sum(axis=1, keepdims=True)
                s = f1_score(yt, aj.argmax(axis=1), average='macro')
                if s > base:
                    base, esc = s, t
    return esc, float(base)


oc, pc = oof_clf()
print('sanity HGB (esperado ~0.467):', round(float(f1_score(y, oc, average='macro')), 4))
orr = oof_reg()
print('regressor sozinho:', round(float(f1_score(y, orr, average='macro')), 4))
maxp = pc.max(axis=1)
best_t, best_f = 0.5, -1.0
for t in np.arange(0.05, 0.65, 0.05):
    f = f1_score(y, np.where(maxp >= t, oc, orr), average='macro')
    if f > best_f:
        best_f, best_t = float(f), round(float(t), 2)
print('stack t*:', best_t, round(best_f, 4))
esc, f1e = tune_escalas(pc, y.values)
print('escalas:', round(f1e, 4))
aj = pc * esc
aj = aj / aj.sum(axis=1, keepdims=True)
pred_s = np.array([CLASSES[i] for i in aj.argmax(axis=1)])
fin = np.maximum(np.where(aj.max(axis=1) >= best_t, pred_s, orr), fmin_tr)
print('FINAL (esperado ~0.47):', round(float(f1_score(y, fin, average='macro')), 4))
np.savez(os.path.join(DATA_DIR, 'combined_oof.npz'), oof=fin, clf=oc, proba=pc)
json.dump({'t_star': best_t, 'escalas': [round(float(v), 3) for v in esc]},
          open(os.path.join(DATA_DIR, 'combined_cfg.json'), 'w'))
print('OOF guardadas para o NB06')

## XGBoost como classificador base

O ambiente passou a ter XGBoost 3.4.1. Usamos uma configuração e um protocolo: 500 árvores, profundidade 6, learning rate 0.05, subsample e colsample 0.8, com pesos de amostra balanceados. O XGBoost não tem `class_weight` para multiclasse, por isso os pesos entram por `sample_weight`.

Há duas notas técnicas. O treino usa uma matriz numpy porque o XGBoost recusa nomes de colunas com `<`, e há one-hots como `filho < 25 anos`. A regra de adoção é simples: o XGBoost só substitui o modelo congelado se a validação cruzada der pelo menos 0.48 com margem.


In [ ]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

Xv = X.values
sw = compute_sample_weight(class_weight="balanced", y=y)
f1s, oof_x, proba_x = [], np.empty(len(y), dtype=int), np.zeros((len(y), len(CLASSES)))
for tr, va in FOLDS:
    m = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                      random_state=RANDOM_STATE, n_jobs=-1)
    m.fit(Xv[tr], y.iloc[tr], sample_weight=sw[tr])
    oof_x[va] = m.predict(Xv[va])
    proba_x[va] = m.predict_proba(Xv[va])
    f1s.append(f1_score(y.iloc[va], oof_x[va], average="macro"))
f1s = np.array(f1s)
print("XGB (esperado ~0.474):", round(float(f1s.mean()), 4), "+/-", round(float(f1s.std()), 4))
np.savez(os.path.join(DATA_DIR, "xgb_oof.npz"), oof=oof_x, proba=proba_x)


## Ensemble de HGB e XGBoost

O primeiro teste é barato e usa as previsões out-of-fold já guardadas: a média ponderada dos dois classificadores. Se a média piorar o resultado, o ensemble é descartado e o XGBoost passa a classificador base da receita completa. A receita repete os mesmos passos da secção anterior com o novo motor.


In [ ]:
h = np.load(os.path.join(DATA_DIR, "combined_oof.npz"))
for w in [0.25, 0.5, 0.75]:
    pe = w * h["proba"] + (1 - w) * proba_x
    print("ensemble HGB%.2f/XGB%.2f:" % (w, 1 - w),
          round(float(f1_score(y, pe.argmax(axis=1), average="macro")), 4))

def oof_reg_xgb(Xm):
    o = np.empty(len(y))
    for tr, va in FOLDS:
        m = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
        m.fit(Xm[tr], y.iloc[tr])
        o[va] = m.predict(Xm[va])
    return np.clip(np.round(o), 0, 15).astype(int)


reg_x = oof_reg_xgb(Xv)
maxp_x = proba_x.max(axis=1)
best_tx, best_fx = 0.5, -1.0
for t in np.arange(0.05, 0.65, 0.05):
    f = f1_score(y, np.where(maxp_x >= t, oof_x, reg_x), average="macro")
    if f > best_fx:
        best_fx, best_tx = float(f), round(float(t), 2)
print("stack-XGB t*:", best_tx, round(best_fx, 4))
esc_x = np.ones(proba_x.shape[1])
base_x = f1_score(y, proba_x.argmax(axis=1), average="macro")
for _ in range(3):
    for k in range(proba_x.shape[1]):
        for g in np.arange(0.5, 2.05, 0.25):
            t = esc_x.copy()
            t[k] = g
            ajx = proba_x * t
            ajx = ajx / ajx.sum(axis=1, keepdims=True)
            s = f1_score(y, ajx.argmax(axis=1), average="macro")
            if s > base_x:
                base_x, esc_x = s, t
ajx = proba_x * esc_x
ajx = ajx / ajx.sum(axis=1, keepdims=True)
pred_sx = np.array([CLASSES[i] for i in ajx.argmax(axis=1)])
fin_x = np.maximum(np.where(ajx.max(axis=1) >= best_tx, pred_sx, reg_x), fmin_tr)
print("XGB FINAL:", round(float(f1_score(y, fin_x, average="macro")), 4))
np.savez(os.path.join(DATA_DIR, "xgb_final_oof.npz"), oof=fin_x)
json.dump({"t_star": best_tx, "escalas": [round(float(v), 3) for v in esc_x]},
          open(os.path.join(DATA_DIR, "xgb_final_cfg.json"), "w"))


## Submissão com HGB

Treinamos o modelo em todo o treino e geramos `sub05_combined.csv`. A regra de submissão é a mesma de sempre: só se o resultado final bater o `sub04` do Notebook 03, que tem 0.475, com margem. Um empate fica congelado e não gasta uma submissão.


In [ ]:
cfg = json.load(open(os.path.join(DATA_DIR, 'combined_cfg.json')))
esc = np.array(cfg['escalas'])
best_t = cfg['t_star']
clf = HistGradientBoostingClassifier(class_weight='balanced', random_state=RANDOM_STATE).fit(X, y)
reg = HistGradientBoostingRegressor(random_state=RANDOM_STATE).fit(X, y)
pt = clf.predict_proba(Xte) * esc
pt = pt / pt.sum(axis=1, keepdims=True)
pc_t = np.array([CLASSES[i] for i in pt.argmax(axis=1)])
pr_t = np.clip(np.round(reg.predict(Xte)), 0, 15).astype(int)
pred = np.maximum(np.where(pt.max(axis=1) >= best_t, pc_t, pr_t), fmin_te)
exemplo = pd.read_csv(os.path.join(DATA_DIR, 'example_submission_10.csv'))
assert len(exemplo) == len(pred) == 3655
sub = exemplo.copy()
sub[TARGET] = pred
sub.to_csv(os.path.join(SUBM_DIR, 'sub05_combined.csv'), index=False)
print('sub05_combined.csv gravada (submeter MANUALMENTE e só pela regra acima)')
print(pd.Series(pred).value_counts().sort_index().to_dict())

## Submissão com XGBoost

A mesma regra se aplica ao XGBoost: gera-se `sub06_xgb.csv` localmente e só se submete se bater o melhor resultado confirmado.


In [ ]:
cfgx = json.load(open(os.path.join(DATA_DIR, "xgb_final_cfg.json")))
escx = np.array(cfgx["escalas"])
tx = cfgx["t_star"]
sw_full = compute_sample_weight(class_weight="balanced", y=y)
clfx = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                    random_state=RANDOM_STATE, n_jobs=-1).fit(X.values, y, sample_weight=sw_full)
regx = HistGradientBoostingRegressor(random_state=RANDOM_STATE).fit(X.values, y)
Xtev = Xte.values
ptx = clfx.predict_proba(Xtev) * escx
ptx = ptx / ptx.sum(axis=1, keepdims=True)
pcx = np.array([CLASSES[i] for i in ptx.argmax(axis=1)])
prx = np.clip(np.round(regx.predict(Xtev)), 0, 15).astype(int)
predx = np.maximum(np.where(ptx.max(axis=1) >= tx, pcx, prx), fmin_te)
subx = exemplo.copy()
subx[TARGET] = predx
subx.to_csv(os.path.join(SUBM_DIR, "sub06_xgb.csv"), index=False)
print("sub06_xgb.csv gravada (submeter MANUALMENTE e só pela regra)")
print(pd.Series(predx).value_counts().sort_index().to_dict())


## Conclusões

- A verificação de sanidade do HGB dá 0.4674. A base nova está válida.

- O XGBoost base dá 0.4740 e é o melhor modelo isolado. O ensemble de HGB e XGBoost dilui o resultado, com 0.472 a 0.473, porque os erros dos dois modelos são correlacionados. O XGBoost passa a base em vez de ser mediado.

- A receita final com XGBoost, stacking com `t*=0.45`, escalas e correção chega a 0.4841. Bate o `sub04` de 0.475 com margem e é adotada. A distribuição fica mais equilibrada, com 284 previsões na classe 11 e 382 na classe 13.

- Submeter `sub06_xgb.csv` manualmente. A expectativa pública honesta é cerca de 0.47, porque o limiar e as escalas foram ajustados na previsão out-of-fold.
